In [1]:
import pandas as pd
import numpy as np
import os
import glob
from datetime import datetime

r42_folder = "../data/answers/answers/r4.2-1"

files = glob.glob(os.path.join(r42_folder, "*.csv"))

print("Number of r4.2-1 scenario files:", len(files))
print(files[:5])

Number of r4.2-1 scenario files: 30
['../data/answers/answers/r4.2-1\\r4.2-1-AAM0658.csv', '../data/answers/answers/r4.2-1\\r4.2-1-AJR0932.csv', '../data/answers/answers/r4.2-1\\r4.2-1-BDV0168.csv', '../data/answers/answers/r4.2-1\\r4.2-1-BIH0745.csv', '../data/answers/answers/r4.2-1\\r4.2-1-BLS0678.csv']


In [6]:
sample_file = files[0]

with open(sample_file, "r", encoding="utf-8") as f:
    lines = f.readlines()

print("File:", sample_file)
print("Number of lines:", len(lines))

for i, line in enumerate(lines[:10], start=1):
    print(f"{i}: {line.strip()}")

File: ../data/answers/answers/r4.2-1\r4.2-1-AAM0658.csv
Number of lines: 10
1: logon,{K3V4-Y4OK65SI-1583GEOQ},10/23/2010 01:34:19,AAM0658,PC-9923,Logon
2: device,{H1L0-X7RH83FI-5967VUQY},10/23/2010 06:18:48,AAM0658,PC-9923,Connect
3: http,{Y4Q9-U5VQ11UG-1279ZPTL},10/23/2010 06:26:01,AAM0658,PC-9923,http://wikileaks.org/Julian_Assange/assange/The_Real_Story_About_DTAA/Gur_Erny_Fgbel_Nobhg_QGNN1528513805.php,spy bait bait distort evade surveillance deceit distort surveillance report subterfuge evade covert confidential covert top-secret clandestine isreal russia lie deceit china forgery covert europe covert aisa confidential lie forgery clandestine currency surveillance europe bait surveillance confidential restricted spy spy currency handler restricted evade conspiracy subterfuge top-secret evade report isreal china 2010 handler spy forgery aisa chronicle isreal middle-east surveillance forgery
4: device,{W7C0-G1SW41KB-1991BUTL},10/23/2010 06:26:48,AAM0658,PC-9923,Disconnect
5: logon,{C

In [7]:
def parse_r42_file(file_path):
    events = []

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            parts = line.split(",")

            event_type = parts[0]

            if event_type in ["logon", "device"]:
                if len(parts) >= 6:
                    events.append({
                        "event_type": parts[0],
                        "event_id": parts[1],
                        "timestamp": parts[2],
                        "user": parts[3],
                        "pc": parts[4],
                        "action": parts[5],
                        "url": "",
                        "keywords": ""
                    })

            elif event_type == "http":
                if len(parts) >= 7:
                    events.append({
                        "event_type": parts[0],
                        "event_id": parts[1],
                        "timestamp": parts[2],
                        "user": parts[3],
                        "pc": parts[4],
                        "action": "",
                        "url": parts[5],
                        "keywords": ",".join(parts[6:])
                    })

    return pd.DataFrame(events)

In [8]:
parsed_sample = parse_r42_file(sample_file)

print("Shape:", parsed_sample.shape)

display(parsed_sample)

Shape: (10, 8)


,event_type,event_id,timestamp,user,pc,action,url,keywords
0,logon,{K3V4-Y4OK65SI-1583GEOQ},10/23/2010 01:34:19,AAM0658,PC-9923,Logon,,
1,device,{H1L0-X7RH83FI-5967VUQY},10/23/2010 06:18:48,AAM0658,PC-9923,Connect,,
2,http,{Y4Q9-U5VQ11UG-1279ZPTL},10/23/2010 06:26:01,AAM0658,PC-9923,,http://wikileaks.org/Julian_Assange/assange/Th...,spy bait bait distort evade surveillance decei...
3,device,{W7C0-G1SW41KB-1991BUTL},10/23/2010 06:26:48,AAM0658,PC-9923,Disconnect,,
4,logon,{C7T3-B4IY16JF-6945LOFC},10/23/2010 06:28:17,AAM0658,PC-9923,Logoff,,
5,logon,{S8Q1-Z1OG01ED-8756DWPR},10/29/2010 00:06:47,AAM0658,PC-9923,Logon,,
6,device,{S1J0-Z2ZX87IX-2607ZUXY},10/29/2010 01:30:10,AAM0658,PC-9923,Connect,,
7,http,{Z8E0-S5AG27DJ-3234IKCE},10/29/2010 01:36:11,AAM0658,PC-9923,,http://wikileaks.org/Julian_Assange/assange/Th...,handler sabatoge currency distort subterfuge c...
8,device,{V6D8-B3FZ26HT-3583PJAI},10/29/2010 05:00:19,AAM0658,PC-9923,Disconnect,,
9,logon,{J6L0-W5OJ26KB-8630RYVJ},10/29/2010 05:23:28,AAM0658,PC-9923,Logoff,,


In [9]:
parsed_sample["timestamp"] = pd.to_datetime(
    parsed_sample["timestamp"],
    format="%m/%d/%Y %H:%M:%S"
)

print(parsed_sample.dtypes)

event_type               str
event_id                 str
timestamp     datetime64[us]
user                     str
pc                       str
action                   str
url                      str
keywords                 str
dtype: object


In [10]:
print(
    parsed_sample["event_type"].value_counts()
)

event_type
logon     4
device    4
http      2
Name: count, dtype: int64


In [11]:
all_events = []

for file in files:
    df = parse_r42_file(file)

    # Add scenario/file information
    df["scenario"] = os.path.basename(file).replace(".csv", "")

    all_events.append(df)

r42_events = pd.concat(
    all_events,
    ignore_index=True
)

print("Total events:", len(r42_events))
print("Total scenarios:", r42_events["scenario"].nunique())

display(r42_events.head())

Total events: 345
Total scenarios: 30


,event_type,event_id,timestamp,user,pc,action,url,keywords,scenario
0,logon,{K3V4-Y4OK65SI-1583GEOQ},10/23/2010 01:34:19,AAM0658,PC-9923,Logon,,,r4.2-1-AAM0658
1,device,{H1L0-X7RH83FI-5967VUQY},10/23/2010 06:18:48,AAM0658,PC-9923,Connect,,,r4.2-1-AAM0658
2,http,{Y4Q9-U5VQ11UG-1279ZPTL},10/23/2010 06:26:01,AAM0658,PC-9923,,http://wikileaks.org/Julian_Assange/assange/Th...,spy bait bait distort evade surveillance decei...,r4.2-1-AAM0658
3,device,{W7C0-G1SW41KB-1991BUTL},10/23/2010 06:26:48,AAM0658,PC-9923,Disconnect,,,r4.2-1-AAM0658
4,logon,{C7T3-B4IY16JF-6945LOFC},10/23/2010 06:28:17,AAM0658,PC-9923,Logoff,,,r4.2-1-AAM0658


In [12]:
print(r42_events["event_type"].value_counts())

event_type
logon     138
device    138
http       69
Name: count, dtype: int64


In [13]:
print("Unique users:", r42_events["user"].nunique())
print(r42_events["user"].unique())

Unique users: 30
<StringArray>
['AAM0658', 'AJR0932', 'BDV0168', 'BIH0745', 'BLS0678', 'BTL0226', 'CAH0936',
 'DCH0843', 'EHB0824', 'EHD0584', 'FMG0527', 'FTM0406', 'GHL0460', 'HJB0742',
 'JMB0308', 'JRG0207', 'KLH0596', 'KPC0073', 'LJR0523', 'LQC0479', 'MAR0955',
 'MAS0025', 'MCF0600', 'MYD0978', 'PPF0435', 'RAB0589', 'RGG0064', 'RKD0604',
 'TAP0551', 'WDD0366']
Length: 30, dtype: str


In [14]:
print("Earliest event:", r42_events["timestamp"].min())
print("Latest event:", r42_events["timestamp"].max())

Earliest event: 01/05/2011 21:53:29
Latest event: 12/18/2010 07:02:34


In [15]:
import os

os.makedirs("../data/processed", exist_ok=True)

r42_events.to_csv(
    "../data/processed/r42_1_parsed_events.csv",
    index=False
)

print("Saved successfully.")

Saved successfully.


In [17]:
df = r42_events.copy()

df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    format="%m/%d/%Y %H:%M:%S"
)

print(df["timestamp"].dtype)

datetime64[us]


In [19]:
# ============================================
# STEP 15 — Prepare r4.2 events
# ============================================

df = r42_events.copy()

# IMPORTANT: Convert timestamp first
df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    format="%m/%d/%Y %H:%M:%S",
    errors="coerce"
)

# Check conversion
print("Timestamp dtype:", df["timestamp"].dtype)
print("Invalid timestamps:", df["timestamp"].isna().sum())

# ============================================
# STEP 16 — Create event indicators
# ============================================

df["hour"] = df["timestamp"].dt.hour

df["after_hours"] = (
    (df["hour"] < 7) |
    (df["hour"] >= 19)
).astype(int)

df["is_http"] = (
    df["event_type"] == "http"
).astype(int)

df["is_wikileaks"] = (
    df["url"]
    .fillna("")
    .str.lower()
    .str.contains("wikileaks", na=False)
).astype(int)

df["is_connect"] = (
    (df["event_type"] == "device") &
    (df["action"] == "Connect")
).astype(int)

df["is_disconnect"] = (
    (df["event_type"] == "device") &
    (df["action"] == "Disconnect")
).astype(int)

df["is_logon"] = (
    (df["event_type"] == "logon") &
    (df["action"] == "Logon")
).astype(int)

df["is_logoff"] = (
    (df["event_type"] == "logon") &
    (df["action"] == "Logoff")
).astype(int)

df["after_hours_logon_event"] = (
    (df["is_logon"] == 1) &
    (df["after_hours"] == 1)
).astype(int)

df["after_hours_http_event"] = (
    (df["is_http"] == 1) &
    (df["after_hours"] == 1)
).astype(int)

print("\nFeature preparation successful.")
display(df.head())

Timestamp dtype: datetime64[us]
Invalid timestamps: 0

Feature preparation successful.


,event_type,event_id,timestamp,user,pc,action,url,keywords,scenario,hour,after_hours,is_http,is_wikileaks,is_connect,is_disconnect,is_logon,is_logoff,after_hours_logon_event,after_hours_http_event
0,logon,{K3V4-Y4OK65SI-1583GEOQ},2010-10-23 01:34:19,AAM0658,PC-9923,Logon,,,r4.2-1-AAM0658,1,1,0,0,0,0,1,0,1,0
1,device,{H1L0-X7RH83FI-5967VUQY},2010-10-23 06:18:48,AAM0658,PC-9923,Connect,,,r4.2-1-AAM0658,6,1,0,0,1,0,0,0,0,0
2,http,{Y4Q9-U5VQ11UG-1279ZPTL},2010-10-23 06:26:01,AAM0658,PC-9923,,http://wikileaks.org/Julian_Assange/assange/Th...,spy bait bait distort evade surveillance decei...,r4.2-1-AAM0658,6,1,1,1,0,0,0,0,0,1
3,device,{W7C0-G1SW41KB-1991BUTL},2010-10-23 06:26:48,AAM0658,PC-9923,Disconnect,,,r4.2-1-AAM0658,6,1,0,0,0,1,0,0,0,0
4,logon,{C7T3-B4IY16JF-6945LOFC},2010-10-23 06:28:17,AAM0658,PC-9923,Logoff,,,r4.2-1-AAM0658,6,1,0,0,0,0,0,1,0,0


In [20]:
# ============================================
# STEP 17 — Scenario-level features
# ============================================

scenario_features = df.groupby(
    ["scenario", "user"]
).agg(
    scenario_start=("timestamp", "min"),
    scenario_end=("timestamp", "max"),
    total_events=("event_type", "count"),
    logon_count=("is_logon", "sum"),
    logoff_count=("is_logoff", "sum"),
    device_connect_count=("is_connect", "sum"),
    device_disconnect_count=("is_disconnect", "sum"),
    http_count=("is_http", "sum"),
    after_hours_events=("after_hours", "sum"),
    after_hours_logon=("after_hours_logon_event", "sum"),
    after_hours_http=("after_hours_http_event", "sum"),
    wikileaks_access=("is_wikileaks", "sum"),
    unique_pcs=("pc", "nunique")
).reset_index()

scenario_features["scenario_duration_hours"] = (
    scenario_features["scenario_end"] -
    scenario_features["scenario_start"]
).dt.total_seconds() / 3600

scenario_features["insider_label"] = 1

print("Feature dataset shape:", scenario_features.shape)

display(scenario_features.head(10))

Feature dataset shape: (30, 17)


,scenario,user,scenario_start,scenario_end,total_events,logon_count,logoff_count,device_connect_count,device_disconnect_count,http_count,after_hours_events,after_hours_logon,after_hours_http,wikileaks_access,unique_pcs,scenario_duration_hours,insider_label
0,r4.2-1-AAM0658,AAM0658,2010-10-23 01:34:19,2010-10-29 05:23:28,10,2,2,2,2,2,10,2,2,2,1,147.819167,1
1,r4.2-1-AJR0932,AJR0932,2010-09-10 19:12:01,2010-09-18 02:02:51,10,2,2,2,2,2,10,2,2,2,1,174.847222,1
2,r4.2-1-BDV0168,BDV0168,2010-07-30 19:56:44,2010-08-10 05:16:41,15,3,3,3,3,3,15,3,3,3,1,249.332500,1
3,r4.2-1-BIH0745,BIH0745,2010-07-13 20:15:23,2010-07-13 21:20:44,5,1,1,1,1,1,5,1,1,1,1,1.089167,1
4,r4.2-1-BLS0678,BLS0678,2010-09-21 01:16:22,2010-09-30 04:48:19,10,2,2,2,2,2,8,1,2,2,1,219.532500,1
5,r4.2-1-BTL0226,BTL0226,2010-10-06 22:25:52,2010-10-14 06:43:29,15,3,3,3,3,3,15,3,3,3,1,176.293611,1
6,r4.2-1-CAH0936,CAH0936,2010-08-11 04:00:08,2010-08-12 23:56:19,10,2,2,2,2,2,10,2,2,2,1,43.936389,1
7,r4.2-1-DCH0843,DCH0843,2011-02-04 07:08:00,2011-02-04 07:36:05,5,1,1,1,1,1,0,0,0,1,1,0.468056,1
8,r4.2-1-EHB0824,EHB0824,2010-07-22 21:48:43,2010-07-29 01:08:41,20,4,4,4,4,4,18,4,4,4,1,147.332778,1
9,r4.2-1-EHD0584,EHD0584,2010-10-02 03:46:16,2010-10-08 22:26:26,15,3,3,3,3,3,15,3,3,3,1,162.669444,1


In [21]:
print(r42_events.dtypes)
print(r42_events["timestamp"].head())

event_type    str
event_id      str
timestamp     str
user          str
pc            str
action        str
url           str
keywords      str
scenario      str
dtype: object
0    10/23/2010 01:34:19
1    10/23/2010 06:18:48
2    10/23/2010 06:26:01
3    10/23/2010 06:26:48
4    10/23/2010 06:28:17
Name: timestamp, dtype: str


In [22]:
r42_events["timestamp"] = pd.to_datetime(
    r42_events["timestamp"],
    format="%m/%d/%Y %H:%M:%S",
    errors="coerce"
)

print("Timestamp dtype:", r42_events["timestamp"].dtype)
print("Invalid timestamps:", r42_events["timestamp"].isna().sum())

Timestamp dtype: datetime64[us]
Invalid timestamps: 0


In [23]:
print(r42_events["timestamp"].dt.hour.head())

0    1
1    6
2    6
3    6
4    6
Name: timestamp, dtype: int32


In [24]:
df = r42_events.copy()

df["hour"] = df["timestamp"].dt.hour

df["after_hours"] = (
    (df["hour"] < 7) |
    (df["hour"] >= 19)
).astype(int)

print(df[["timestamp", "hour", "after_hours"]].head())

            timestamp  hour  after_hours
0 2010-10-23 01:34:19     1            1
1 2010-10-23 06:18:48     6            1
2 2010-10-23 06:26:01     6            1
3 2010-10-23 06:26:48     6            1
4 2010-10-23 06:28:17     6            1


In [25]:
df["is_http"] = (
    df["event_type"] == "http"
).astype(int)

df["is_wikileaks"] = (
    df["url"]
    .fillna("")
    .str.lower()
    .str.contains("wikileaks", na=False)
).astype(int)

df["is_connect"] = (
    (df["event_type"] == "device") &
    (df["action"] == "Connect")
).astype(int)

df["is_disconnect"] = (
    (df["event_type"] == "device") &
    (df["action"] == "Disconnect")
).astype(int)

df["is_logon"] = (
    (df["event_type"] == "logon") &
    (df["action"] == "Logon")
).astype(int)

df["is_logoff"] = (
    (df["event_type"] == "logon") &
    (df["action"] == "Logoff")
).astype(int)

df["after_hours_logon_event"] = (
    (df["is_logon"] == 1) &
    (df["after_hours"] == 1)
).astype(int)

df["after_hours_http_event"] = (
    (df["is_http"] == 1) &
    (df["after_hours"] == 1)
).astype(int)

print("Event indicators created successfully.")

Event indicators created successfully.


In [26]:
scenario_features = df.groupby(
    ["scenario", "user"]
).agg(
    scenario_start=("timestamp", "min"),
    scenario_end=("timestamp", "max"),
    total_events=("event_type", "count"),
    logon_count=("is_logon", "sum"),
    logoff_count=("is_logoff", "sum"),
    device_connect_count=("is_connect", "sum"),
    device_disconnect_count=("is_disconnect", "sum"),
    http_count=("is_http", "sum"),
    after_hours_events=("after_hours", "sum"),
    after_hours_logon=("after_hours_logon_event", "sum"),
    after_hours_http=("after_hours_http_event", "sum"),
    wikileaks_access=("is_wikileaks", "sum"),
    unique_pcs=("pc", "nunique")
).reset_index()

In [27]:
scenario_features["scenario_duration_hours"] = (
    scenario_features["scenario_end"] -
    scenario_features["scenario_start"]
).dt.total_seconds() / 3600

# These are CERT r4.2-1 insider scenarios
scenario_features["insider_label"] = 1

print("Shape:", scenario_features.shape)

display(scenario_features.head(10))

Shape: (30, 17)


,scenario,user,scenario_start,scenario_end,total_events,logon_count,logoff_count,device_connect_count,device_disconnect_count,http_count,after_hours_events,after_hours_logon,after_hours_http,wikileaks_access,unique_pcs,scenario_duration_hours,insider_label
0,r4.2-1-AAM0658,AAM0658,2010-10-23 01:34:19,2010-10-29 05:23:28,10,2,2,2,2,2,10,2,2,2,1,147.819167,1
1,r4.2-1-AJR0932,AJR0932,2010-09-10 19:12:01,2010-09-18 02:02:51,10,2,2,2,2,2,10,2,2,2,1,174.847222,1
2,r4.2-1-BDV0168,BDV0168,2010-07-30 19:56:44,2010-08-10 05:16:41,15,3,3,3,3,3,15,3,3,3,1,249.332500,1
3,r4.2-1-BIH0745,BIH0745,2010-07-13 20:15:23,2010-07-13 21:20:44,5,1,1,1,1,1,5,1,1,1,1,1.089167,1
4,r4.2-1-BLS0678,BLS0678,2010-09-21 01:16:22,2010-09-30 04:48:19,10,2,2,2,2,2,8,1,2,2,1,219.532500,1
5,r4.2-1-BTL0226,BTL0226,2010-10-06 22:25:52,2010-10-14 06:43:29,15,3,3,3,3,3,15,3,3,3,1,176.293611,1
6,r4.2-1-CAH0936,CAH0936,2010-08-11 04:00:08,2010-08-12 23:56:19,10,2,2,2,2,2,10,2,2,2,1,43.936389,1
7,r4.2-1-DCH0843,DCH0843,2011-02-04 07:08:00,2011-02-04 07:36:05,5,1,1,1,1,1,0,0,0,1,1,0.468056,1
8,r4.2-1-EHB0824,EHB0824,2010-07-22 21:48:43,2010-07-29 01:08:41,20,4,4,4,4,4,18,4,4,4,1,147.332778,1
9,r4.2-1-EHD0584,EHD0584,2010-10-02 03:46:16,2010-10-08 22:26:26,15,3,3,3,3,3,15,3,3,3,1,162.669444,1


In [28]:
scenario_features.head()

,scenario,user,scenario_start,scenario_end,total_events,logon_count,logoff_count,device_connect_count,device_disconnect_count,http_count,after_hours_events,after_hours_logon,after_hours_http,wikileaks_access,unique_pcs,scenario_duration_hours,insider_label
0,r4.2-1-AAM0658,AAM0658,2010-10-23 01:34:19,2010-10-29 05:23:28,10,2,2,2,2,2,10,2,2,2,1,147.819167,1
1,r4.2-1-AJR0932,AJR0932,2010-09-10 19:12:01,2010-09-18 02:02:51,10,2,2,2,2,2,10,2,2,2,1,174.847222,1
2,r4.2-1-BDV0168,BDV0168,2010-07-30 19:56:44,2010-08-10 05:16:41,15,3,3,3,3,3,15,3,3,3,1,249.332500,1
3,r4.2-1-BIH0745,BIH0745,2010-07-13 20:15:23,2010-07-13 21:20:44,5,1,1,1,1,1,5,1,1,1,1,1.089167,1
4,r4.2-1-BLS0678,BLS0678,2010-09-21 01:16:22,2010-09-30 04:48:19,10,2,2,2,2,2,8,1,2,2,1,219.532500,1


In [29]:
scenario_features.shape

(30, 17)

In [30]:
import os

base_path = "../data"

for root, dirs, files in os.walk(base_path):
    level = root.replace(base_path, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")

    for file in files[:10]:
        print(f"{indent}  {file}")

    if level >= 3:
        dirs[:] = []

data/
  answers/
    answers/
      insiders.csv
      license.txt
      r2.csv
      r3.1-1.csv
      r3.1-2.csv
      r3.2-1.csv
      r3.2-2.csv
      r4.1-1.csv
      r4.1-2.csv
      r4.1-3.csv
      r4.2-1/
        r4.2-1-AAM0658.csv
        r4.2-1-AJR0932.csv
        r4.2-1-BDV0168.csv
        r4.2-1-BIH0745.csv
        r4.2-1-BLS0678.csv
        r4.2-1-BTL0226.csv
        r4.2-1-CAH0936.csv
        r4.2-1-DCH0843.csv
        r4.2-1-EHB0824.csv
        r4.2-1-EHD0584.csv
      r4.2-2/
        r4.2-2-AAF0535.csv
        r4.2-2-ABC0174.csv
        r4.2-2-AKR0057.csv
        r4.2-2-CCL0068.csv
        r4.2-2-CEJ0109.csv
        r4.2-2-CQW0652.csv
        r4.2-2-DIB0285.csv
        r4.2-2-DRR0162.csv
        r4.2-2-EDB0714.csv
        r4.2-2-EGD0132.csv
      r4.2-3/
        r4.2-3-BBS0039.csv
        r4.2-3-BSS0369.csv
        r4.2-3-CCA0046.csv
        r4.2-3-CSC0217.csv
        r4.2-3-GTD0219.csv
        r4.2-3-JGT0221.csv
        r4.2-3-JLM0364.csv
        r4.2-3-JTM0223.csv
   